# 26. The Gate Automation & Damage Detection Problem

## Tier 2: The Classic Heuristic (Python Implementation)

### Goal
Implement a priority-based greedy algorithm for real-time sensor monitoring decisions.

### Key assumptions
- Sensors can be prioritized based on multiple factors
- Real-time decisions can be made using weighted scoring
- Priority queue enables efficient sensor selection
- Budget constraints limit simultaneous monitoring

### Approach (step-by-step)
1. Define priority scoring function
2. Create data structures for gates and sensors
3. Implement priority queue for sensor selection
4. Develop greedy algorithm for monitoring decisions
5. Add budget management logic
6. Simulate algorithm performance
7. Compare with optimal solution

### Concrete example (from the source)
4-gate facility over 48 hours:
- Monitoring budget: $200/hour
- Algorithm selects top 3 sensors for monitoring
- Achieves 94% system reliability within budget
- Dynamic adjustment during peak hours

In [1]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import heapq
from dataclasses import dataclass
from typing import List, Dict, Tuple
import random
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

print('Priority-Based Sensor Monitoring Algorithm')
print('=' * 50)

Priority-Based Sensor Monitoring Algorithm


Priority-Based Sensor Monitoring Algorithm


Priority-Based Sensor Monitoring Algorithm


Priority-Based Sensor Monitoring Algorithm


Priority-Based Sensor Monitoring Algorithm


### Data Structures

In [ ]:
@dataclass
class SensorState:
    gate_id: int
    sensor_type: str
    health_index: float = 1.0
    failure_count: int = 0
    environmental_factor: float = 1.0

@dataclass
class GateState:
    gate_id: int
    demand_level: float = 1.0
    is_operational: bool = True
    total_vehicles_processed: int = 0

@dataclass
class PrioritySensor:
    priority: float
    gate_id: int
    sensor_type: str
    
    def __lt__(self, other):
        return self.priority > other.priority  # Max heap

@dataclass
class HeuristicInstance:
    gates: List[GateState]
    sensors: List[SensorState]
    time_periods: int
    monitoring_budget_per_hour: float
    sensor_costs: Dict[str, float]
    gate_downtime_costs: Dict[int, float]
    demand_patterns: Dict[int, List[float]]

### Create the Example Instance

In [ ]:
def create_heuristic_example():
    # Define costs
    sensor_costs = {'photo_eye': 50, 'ground_loop': 60, 'safety_beam': 70}
    gate_downtime_costs = {1: 15000, 2: 12000, 3: 10000, 4: 8000}
    
    # Create demand patterns (peak: 8-10 AM, 5-7 PM)
    demand_patterns = {}
    for gate_id in range(1, 5):
        base_demand = 150 - (gate_id - 1) * 25
        pattern = []
        for hour in range(48):
            hour_of_day = hour % 24
            if 8 <= hour_of_day <= 10 or 17 <= hour_of_day <= 19:
                peak_factor = 1.8
            elif 6 <= hour_of_day <= 11 or 16 <= hour_of_day <= 21:
                peak_factor = 1.3
            else:
                peak_factor = 0.7
            pattern.append(base_demand * peak_factor)
        demand_patterns[gate_id] = pattern
    
    # Initialize gates
    gates = []
    for gate_id in range(1, 5):
        gates.append(GateState(
            gate_id=gate_id,
            demand_level=demand_patterns[gate_id][0] / 100
        ))
    
    # Initialize sensors
    sensors = []
    sensor_configs = [
        (1, 'photo_eye', 0.95, 5, 1.2),      # High priority
        (1, 'ground_loop', 0.85, 2, 1.0),
        (1, 'safety_beam', 0.90, 1, 1.1),
        (2, 'photo_eye', 0.88, 3, 1.0),
        (2, 'ground_loop', 0.92, 1, 1.3),    # High dust
        (2, 'safety_beam', 0.87, 4, 1.1),    # High priority
        (3, 'photo_eye', 0.93, 1, 0.9),      # Lower stress
        (3, 'ground_loop', 0.89, 2, 1.0),
        (3, 'safety_beam', 0.91, 1, 0.8),
        (4, 'photo_eye', 0.94, 0, 0.8),      # New sensors
        (4, 'ground_loop', 0.96, 0, 0.9),
        (4, 'safety_beam', 0.95, 1, 0.7),
    ]
    
    for gate_id, sensor_type, health, failures, env_factor in sensor_configs:
        sensors.append(SensorState(
            gate_id=gate_id,
            sensor_type=sensor_type,
            health_index=health,
            failure_count=failures,
            environmental_factor=env_factor
        ))
    
    return HeuristicInstance(
        gates=gates,
        sensors=sensors,
        time_periods=48,
        monitoring_budget_per_hour=200,
        sensor_costs=sensor_costs,
        gate_downtime_costs=gate_downtime_costs,
        demand_patterns=demand_patterns
    )

# Create instance
heuristic_instance = create_heuristic_example()
print(f'Heuristic instance created:')
print(f'  {len(heuristic_instance.gates)} gates')
print(f'  {len(heuristic_instance.sensors)} sensors')
print(f'  {heuristic_instance.time_periods} time periods')
print(f'  Budget: ${heuristic_instance.monitoring_budget_per_hour}/hour')

### Priority-Based Algorithm Implementation

In [ ]:
class PriorityBasedSensorMonitoring:
    def __init__(self, instance: HeuristicInstance):
        self.instance = instance
        self.priority_weights = {
            'health': 0.3,
            'failure_history': 0.25,
            'demand': 0.2,
            'environmental': 0.15,
            'criticality': 0.1
        }
        
        self.sensor_criticality = {
            'safety_beam': 3.0,
            'photo_eye': 2.0,
            'ground_loop': 1.0
        }
        
        self.results = {
            'monitoring_schedule': [],
            'system_reliability': [],
            'budget_utilization': [],
            'gate_availabilities': []
        }
    
    def calculate_priority_score(self, sensor: SensorState, gate: GateState, hour: int) -> float:
        # Health component (lower health = higher priority)
        health_score = (1.0 - sensor.health_index) * self.priority_weights['health']
        
        # Failure history component
        failure_score = min(sensor.failure_count / 10.0, 1.0) * self.priority_weights['failure_history']
        
        # Demand component
        current_demand = self.instance.demand_patterns[gate.gate_id][hour] / 150.0
        demand_score = current_demand * self.priority_weights['demand']
        
        # Environmental stress component
        env_score = (sensor.environmental_factor - 0.5) / 1.5 * self.priority_weights['environmental']
        
        # Criticality component
        criticality_score = (self.sensor_criticality[sensor.sensor_type] / 3.0) * self.priority_weights['criticality']
        
        return health_score + failure_score + demand_score + env_score + criticality_score
    
    def select_sensors_to_monitor(self, hour: int) -> List[Tuple[int, str]]:
        # Create priority queue
        priority_queue = []
        
        for sensor in self.instance.sensors:
            gate = self.instance.gates[sensor.gate_id - 1]
            priority = self.calculate_priority_score(sensor, gate, hour)
            
            priority_item = PrioritySensor(
                priority=priority,
                gate_id=sensor.gate_id,
                sensor_type=sensor.sensor_type
            )
            heapq.heappush(priority_queue, priority_item)
        
        # Select sensors within budget
        selected_sensors = []
        remaining_budget = self.instance.monitoring_budget_per_hour
        
        while priority_queue and remaining_budget > 0:
            priority_sensor = heapq.heappop(priority_queue)
            sensor_cost = self.instance.sensor_costs[priority_sensor.sensor_type]
            
            if sensor_cost <= remaining_budget:
                selected_sensors.append((priority_sensor.gate_id, priority_sensor.sensor_type))
                remaining_budget -= sensor_cost
        
        return selected_sensors
    
    def update_states(self, monitored_sensors: List[Tuple[int, str]], hour: int):
        # Update sensor states
        for sensor in self.instance.sensors:
            is_monitored = (sensor.gate_id, sensor.sensor_type) in monitored_sensors
            
            if is_monitored:
                # Monitored sensors: better maintenance
                sensor.health_index = min(1.0, sensor.health_index + 0.01)
                failure_prob = 0.005 * sensor.environmental_factor
            else:
                # Unmonitored sensors: faster degradation
                degradation = 0.02 * sensor.environmental_factor
                sensor.health_index = max(0.0, sensor.health_index - degradation)
                failure_prob = 0.02 * sensor.environmental_factor
            
            # Check for failure
            if random.random() < failure_prob and sensor.health_index < 0.3:
                sensor.failure_count += 1
                sensor.health_index = max(0.0, sensor.health_index - 0.1)
        
        # Update gate states
        for gate in self.instance.gates:
            gate.demand_level = self.instance.demand_patterns[gate.gate_id][hour] / 100.0
            
            # Check gate sensors
            gate_sensors = [s for s in self.instance.sensors if s.gate_id == gate.gate_id]
            avg_health = np.mean([s.health_index for s in gate_sensors])
            
            if avg_health < 0.2:
                gate.is_operational = False
            else:
                gate.is_operational = True
            
            if gate.is_operational:
                vehicles_processed = int(self.instance.demand_patterns[gate.gate_id][hour])
                gate.total_vehicles_processed += vehicles_processed
    
    def run_simulation(self) -> Dict:
        print('Running Priority-Based Sensor Monitoring simulation...')
        
        for hour in range(self.instance.time_periods):
            # Select sensors to monitor
            monitored_sensors = self.select_sensors_to_monitor(hour)
            
            # Update states
            self.update_states(monitored_sensors, hour)
            
            # Record results
            self.results['monitoring_schedule'].append(monitored_sensors)
            
            # Calculate system reliability
            operational_gates = sum(1 for g in self.instance.gates if g.is_operational)
            reliability = operational_gates / len(self.instance.gates) * 100
            self.results['system_reliability'].append(reliability)
            
            # Calculate budget utilization
            monitoring_cost = sum(self.instance.sensor_costs[s_type] for g_id, s_type in monitored_sensors)
            budget_util = monitoring_cost / self.instance.monitoring_budget_per_hour * 100
            self.results['budget_utilization'].append(budget_util)
            
            # Record gate availabilities
            gate_avail = {g.gate_id: (100 if g.is_operational else 0) for g in self.instance.gates}
            self.results['gate_availabilities'].append(gate_avail)
            
            # Progress indicator
            if (hour + 1) % 12 == 0:
                print(f'  Hour {hour+1:2d}/{self.instance.time_periods}: Reliability = {reliability:.1f}%')
        
        return self.results

### Execute the Algorithm

In [ ]:
# Run the algorithm
pbsm = PriorityBasedSensorMonitoring(heuristic_instance)
results = pbsm.run_simulation()

# Calculate final statistics
final_reliability = np.mean(results['system_reliability'])
final_budget_util = np.mean(results['budget_utilization'])
total_vehicles = sum(g.total_vehicles_processed for g in heuristic_instance.gates)

print(f'\nSIMULATION RESULTS')
print(f'Average System Reliability: {final_reliability:.1f}%')
print(f'Average Budget Utilization: {final_budget_util:.1f}%')
print(f'Total Vehicles Processed: {total_vehicles:,}')
print(f'Simulation Duration: {heuristic_instance.time_periods} hours')

print(f'\nCOMPARISON WITH SOURCE EXAMPLE')
print(f'Source reported reliability: 94.0%')
print(f'Our algorithm achieved: {final_reliability:.1f}%')
print(f'Difference: {final_reliability - 94.0:+.1f}%')

### Visualization and Analysis

In [ ]:
# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Priority-Based Sensor Monitoring Results', fontsize=14, fontweight='bold')

# 1. System Reliability Over Time
ax1 = axes[0, 0]
hours = list(range(len(results['system_reliability'])))
reliability = results['system_reliability']

ax1.plot(hours, reliability, linewidth=2, color='#2ecc71')
ax1.fill_between(hours, reliability, alpha=0.3, color='#2ecc71')
ax1.axhline(y=94, color='red', linestyle='--', alpha=0.7, label='Target (94%)')
ax1.set_title('System Reliability Over Time')
ax1.set_xlabel('Time (Hours)')
ax1.set_ylabel('Reliability (%)')
ax1.set_ylim(0, 105)
ax1.grid(True, alpha=0.3)
ax1.legend()

# 2. Budget Utilization
ax2 = axes[0, 1]
budget_util = results['budget_utilization']

ax2.plot(hours, budget_util, linewidth=2, color='#3498db')
ax2.fill_between(hours, budget_util, alpha=0.3, color='#3498db')
ax2.axhline(y=100, color='red', linestyle='--', alpha=0.7, label='Budget Limit')
ax2.set_title('Budget Utilization')
ax2.set_xlabel('Time (Hours)')
ax2.set_ylabel('Budget Utilization (%)')
ax2.set_ylim(0, 105)
ax2.grid(True, alpha=0.3)
ax2.legend()

# 3. Monitoring Frequency Analysis
ax3 = axes[1, 0]
monitoring_counts = defaultdict(int)
for hour_schedule in results['monitoring_schedule']:
    for gate_id, sensor_type in hour_schedule:
        monitoring_counts[(gate_id, sensor_type)] += 1

# Calculate percentages
monitoring_percentages = {}
for (gate_id, sensor_type), count in monitoring_counts.items():
    monitoring_percentages[(gate_id, sensor_type)] = count / heuristic_instance.time_periods * 100

# Sort by percentage
sorted_sensors = sorted(monitoring_percentages.items(), key=lambda x: x[1], reverse=True)
top_sensors = sorted_sensors[:8]  # Top 8 sensors

sensor_labels = [f'G{g_id}-{s_type}' for (g_id, s_type), _ in top_sensors]
sensor_values = [perc for _, perc in top_sensors]

bars = ax3.bar(sensor_labels, sensor_values, color='#9b59b6')
ax3.set_title('Top 8 Most Monitored Sensors')
ax3.set_ylabel('Monitoring Time (%)')
ax3.set_xticklabels(sensor_labels, rotation=45, ha='right')
ax3.grid(axis='y', alpha=0.3)

# 4. Gate Availability Comparison
ax4 = axes[1, 1]
gate_availabilities = {i: [] for i in range(1, 5)}
for hour_data in results['gate_availabilities']:
    for gate_id, availability in hour_data.items():
        gate_availabilities[gate_id].append(availability)

avg_availabilities = [np.mean(gate_availabilities[i]) for i in range(1, 5)]
gate_labels = [f'Gate {i}' for i in range(1, 5)]

bars = ax4.bar(gate_labels, avg_availabilities, 
               color=['#2ecc71' if a > 95 else '#f39c12' if a > 90 else '#e74c3c' for a in avg_availabilities])
ax4.set_title('Average Gate Availability')
ax4.set_ylabel('Availability (%)')
ax4.set_ylim(0, 105)
ax4.grid(axis='y', alpha=0.3)

for bar, avail in zip(bars, avg_availabilities):
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height + 1,
            f'{avail:.1f}%', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print('Heuristic algorithm visualization completed!')

### Performance Analysis

Time complexity: O(|G| × |S| × log(|G| × |S|)) per period
Memory usage: O(|G| × |S| × |T|) for results storage

Scalability projections:
- 10 gates, 30 sensors, 1 week: ~50,000 operations
- 20 gates, 60 sensors, 1 week: ~200,000 operations
- 50 gates, 150 sensors, 1 week: ~1,000,000 operations

### Why This Tier Exists vs Tier 1

**Tier 1 Limitations:**
- Computationally expensive for large instances
- Not suitable for real-time decisions
- Requires complete parameter information

**Tier 2 Advantages:**
- Real-time performance with O(n log n) complexity
- Dynamic adaptation to changing conditions
- Scalable to large facilities
- Easy to implement in production

**Pros:**
- Fast execution for real-time operations
- Handles uncertainty through priority scoring
- Scalable to large networks
- Adaptive to changing conditions

**Cons:**
- No optimality guarantee
- May get stuck in local optima
- Parameter sensitivity
- Limited long-term coordination

**When to use:**
- Real-time operations requiring immediate decisions
- Large-scale facilities with hundreds of gates
- Dynamic environments with changing demand
- Resource-constrained systems
- Production systems requiring robust solutions

This approach provides a practical balance between performance and computational efficiency, serving as a foundation for advanced metaheuristic approaches.